# Multi-Subnet Network Topology & Load Balancer: Corrected Demo

**Version 2: the corrected, working system.** Version 1 (`WK5_Network_Audit_Demo.ipynb`) documents the AI's original draft and the flaws found in it. This notebook applies the fixes and proves them with the four QA tests, live.

| Video segment | Notebook section |
|---|---|
| 0:00 – 1:30 · Architecture & AI verification | 1. Topology, 2. The flaw we caught |
| 1:30 – 3:00 · Live functionality & traffic distribution | 3. Launch, 4. QA 1 round-robin |
| 3:00 – 5:00 · QA checklist, failover & isolation | 5. QA 2 failover, 6. QA 3 isolation, 7. QA 4 health & browser, 8. Logs |
| 5:00 – 6:30 · Conclusion & takeaways | 9. QA summary, 10. Takeaways |

> **How to use this notebook**
> 1. **Test run:** start Docker Desktop, then *Run → Run All Cells*. Every QA cell should end with **PASS**.
> 2. **Before recording:** *Kernel → Restart Kernel and Clear Outputs of All Cells*, then save.
> 3. **While recording:** run one cell at a time with **Shift + Enter** and talk through each result.

## 1. Network topology

```
                    Host / Browser / curl
                            │
                  http://localhost:8080
                            │
  ┌─────────────────────────┴──────────────────────────┐
  │ PUBLIC_SUBNET   172.28.1.0/24                      │
  │                                                    │
  │        nginx  (load balancer)                      │
  │        published port 8080 -> 80                   │
  └─────────────────────────┬──────────────────────────┘
                            │   nginx is dual-homed
                            │
  ┌─────────────────────────┴──────────────────────────┐
  │ PRIVATE_SUBNET  172.28.2.0/24   internal: true     │
  │                                                    │
  │        nginx  (second network interface)           │
  │                                                    │
  │   backend1:8000              backend2:8000         │
  │   SERVER_ID=backend1         SERVER_ID=backend2    │
  │   expose 8000 only           expose 8000 only      │
  │   healthcheck /healthz       healthcheck /healthz  │
  └────────────────────────────────────────────────────┘
```

| Component | Network(s) | Reachable from the host? | Role |
|---|---|---|---|
| `nginx` | public **and** private | Yes, only via port **8080** | Reverse proxy and round-robin load balancer; the single entry point |
| `backend1` | private only | **No** | Python web server, replies `Request handled by backend1` |
| `backend2` | private only | **No** | Python web server, replies `Request handled by backend2` |
| `public_subnet` | `172.28.1.0/24` | | Bridge network facing the host |
| `private_subnet` | `172.28.2.0/24`, `internal: true` | | Isolated backend network with no host ports and no internet access |

Docker assigns each container an address inside its subnet's range when it starts. Section 3 shows the actual addresses.

### Setup: helper functions

**What this cell does:** loads small helper functions used by every test. It sends nothing and changes nothing; it only checks that Docker is running and that port 8080 is free.

- `sh(...)` runs a Docker command from this project folder and prints its output.
- `get(path)` sends one HTTP request to the load balancer and returns the status, body, time taken and headers.
- `send_requests(n)` sends `n` requests and prints a table of which backend answered each one.

In [1]:
import subprocess, json, time, urllib.request, urllib.error, difflib
from pathlib import Path
from collections import Counter

PROJECT = Path.cwd()                    # this notebook's folder = the project folder
BASE_URL = "http://localhost:8080"      # the load balancer: the only published port

def sh(*args, show=True, timeout=240):
    """Run a command in the project folder and (optionally) print its output."""
    r = subprocess.run(list(args), cwd=PROJECT, capture_output=True, text=True,
                       encoding="utf-8", errors="replace", timeout=timeout)
    out = (r.stdout + r.stderr).strip()
    if show:
        print(f"$ {' '.join(args)}\n{out}\n")
    return r.returncode, out

def get(path="/", url=None, timeout=5):
    """One HTTP GET. Returns (status, body, seconds, headers)."""
    start = time.perf_counter()
    try:
        with urllib.request.urlopen(url or BASE_URL + path, timeout=timeout) as r:
            return r.status, r.read().decode().strip(), round(time.perf_counter() - start, 3), r.headers
    except urllib.error.HTTPError as e:
        return e.code, str(e.reason), round(time.perf_counter() - start, 3), e.headers
    except Exception as e:
        return "FAILED", type(e).__name__, round(time.perf_counter() - start, 3), None

def send_requests(n, path="/"):
    """Send n requests, print a table, return the results."""
    results = []
    print(f"{'#':>3}  {'status':<7} {'time':>7}  response")
    for i in range(1, n + 1):
        status, body, secs, _ = get(path)
        results.append((status, body, secs))
        print(f"{i:>3}  {status!s:<7} {secs:>6.3f}s  {body}")
    return results

def container_id(service):
    return sh("docker", "compose", "ps", "-q", service, show=False)[1].strip()

print("Project folder:", PROJECT)
for f in ["compose.yaml", "nginx/nginx.conf", "app.py", "Dockerfile"]:
    print(f"  {f:<18} {'FOUND' if (PROJECT / f).exists() else 'MISSING'}")

rc, version = sh("docker", "info", "--format", "{{.ServerVersion}}", show=False)
print("\nDocker engine:", version if rc == 0 else "NOT RUNNING - start Docker Desktop first")

_, users = sh("docker", "ps", "--filter", "publish=8080", "--format", "{{.Names}}", show=False)
others = [u for u in users.splitlines() if u and not u.startswith(PROJECT.name.lower())]
print("Port 8080:", "in use by " + ", ".join(others) + "  <- stop these first" if others else "free")

Project folder: C:\Users\marky\Documents\AD_BAS\FALL_QTR_27\AD371_APP_ARCH\WK5\multi_subnet_load_balancer
  compose.yaml       FOUND
  nginx/nginx.conf   FOUND
  app.py             FOUND
  Dockerfile         FOUND

Docker engine: 29.4.3
Port 8080: free


**Expected:** all four files `FOUND`, a Docker engine version, and port 8080 `free`.

If port 8080 is in use by `wk5-nginx-1`, the old stack from before the folder move is still running. Run `docker compose down` in the old `WK5` folder, or stop it in Docker Desktop.

## 2. The flaw we caught in the AI's draft

The AI's draft had the right **shape**: no backend ports were published, nginx was the only service on both networks, service names were used instead of IPs, and balancing was round-robin. It never planned for **failure**, though:

| Checklist item | AI's draft | Effect |
|---|---|---|
| Connection timeout | **missing** (60-second default) | A stopped backend made requests hang |
| `proxy_next_upstream` retry | **missing** | Failed requests were not moved to the healthy backend |
| Health checks | nginx answered `/healthz` itself | Reported "healthy" with a backend down |
| Recovery | no `restart:` and no `healthcheck:` | A stopped backend was never noticed |
| Isolation | private network not `internal` | Backends could reach the internet |

**Evidence from version 1:** with `backend1` stopped, **2 of 5 requests timed out**:
```
curl: (28) Operation timed out after 5015 milliseconds with 0 bytes received
Request handled by backend2
curl: (28) Operation timed out after 5006 milliseconds with 0 bytes received
Request handled by backend2
Request handled by backend2
```

**What this cell does:** compares the AI's original `nginx.conf`, saved in `audit_evidence`, with the corrected one. Lines starting with `+` were added; lines starting with `-` were removed.

In [2]:
snapshots = sorted((PROJECT / "audit_evidence").glob("before_restoration_*"))
original = (snapshots[-1] / "nginx" / "nginx.conf").read_text(encoding="utf-8").splitlines()
corrected = (PROJECT / "nginx" / "nginx.conf").read_text(encoding="utf-8").splitlines()

# Show only the real directives (skip comment-only lines) so the changes stand out.
strip = lambda lines: [l for l in lines if l.strip() and not l.strip().startswith("#")]
for line in difflib.unified_diff(strip(original), strip(corrected),
                                 "AI original nginx.conf", "corrected nginx.conf", lineterm="", n=0):
    if not line.startswith("@@"):
        print(line)

--- AI original nginx.conf
+++ corrected nginx.conf
+error_log /var/log/nginx/error.log warn;
+    resolver 127.0.0.11 valid=5s ipv6=off;
+    resolver_timeout 2s;
+    log_format upstreamlog '$remote_addr "$request" $status '
+                           'upstream=$upstream_addr '
+                           'upstream_status=$upstream_status '
+                           'time=${request_time}s';
+    access_log /var/log/nginx/access.log upstreamlog;
-        server backend1:8000;
-        server backend2:8000;
+        zone backend_pool 64k;   # shared memory, required for 'resolve'
+        server backend1:8000 resolve max_fails=1 fail_timeout=5s;
+        server backend2:8000 resolve max_fails=1 fail_timeout=5s;
-        location /healthz {
+        add_header X-Upstream-Server $upstream_addr always;
+        location = /healthz {
+            default_type text/plain;     # one Content-Type header only
+            return 200 "nginx healthy\n";
+        }
+        location = /healthz

**Point out in the video:**
- `+ proxy_connect_timeout 1s;`: give up on an unreachable backend after 1 second instead of 60.
- `+ proxy_next_upstream error timeout http_502 http_503 http_504;`: retry the request on the other backend, so the client never sees the failure.
- `+ max_fails=1 fail_timeout=5s` and `resolve`: take a failed backend out of rotation, and add it back automatically when it returns.
- `- add_header Content-Type text/plain;`: removes the duplicate header bug.

`compose.yaml` was also corrected: the private network is `internal: true`, the backends use `expose: "8000"`, there are health checks that call `/healthz`, nginx waits until the backends are healthy, and `restart: unless-stopped` is set.

## 3. Launch the environment

**What this cell does:**
1. `docker compose down`: removes any previous run, for a clean start.
2. `docker compose up -d --build --wait`: builds the backend image, starts all three containers, and **waits until the health checks pass**.
3. `docker compose ps`: shows the running containers and their ports.

In [3]:
sh("docker", "compose", "down", "--remove-orphans", show=False)
rc, out = sh("docker", "compose", "up", "-d", "--build", "--wait", "--wait-timeout", "120", show=False)
print("docker compose up:", "OK" if rc == 0 else "FAILED\n" + out[-2000:])
_ = sh("docker", "compose", "ps", "--format", "table {{.Service}}\t{{.Status}}\t{{.Ports}}")

docker compose up: OK
$ docker compose ps --format table {{.Service}}	{{.Status}}	{{.Ports}}
SERVICE    STATUS                   PORTS
backend1   Up 7 seconds (healthy)   8000/tcp
backend2   Up 7 seconds (healthy)   8000/tcp
nginx      Up Less than a second    0.0.0.0:8080->80/tcp, [::]:8080->80/tcp



(0,
 'SERVICE    STATUS                   PORTS\nbackend1   Up 7 seconds (healthy)   8000/tcp\nbackend2   Up 7 seconds (healthy)   8000/tcp\nnginx      Up Less than a second    0.0.0.0:8080->80/tcp, [::]:8080->80/tcp')

**Expected:**

| Service | Status | Ports |
|---|---|---|
| `backend1` | Up (**healthy**) | `8000/tcp`, internal only, no `->` |
| `backend2` | Up (**healthy**) | `8000/tcp`, internal only, no `->` |
| `nginx` | Up | `0.0.0.0:8080->80/tcp`, the **only** host port |

`(healthy)` means Docker itself called each backend's `/healthz` and got 200, which the AI's draft never did.

### 3.1 Inspect the live networks

**What this cell does:** reads the networks Docker actually created and lists each one's subnet, whether it is internal, and which containers are attached with their IP addresses.

In [4]:
cfg = json.loads(sh("docker", "compose", "config", "--format", "json", show=False)[1])
net_info = {}
for key in ["public_subnet", "private_subnet"]:
    name = cfg["networks"][key]["name"]
    info = json.loads(sh("docker", "network", "inspect", name, show=False)[1])[0]
    net_info[key] = info
    subnet = info["IPAM"]["Config"][0]["Subnet"]
    print(f"{key:<15} {subnet:<16} internal={info['Internal']}")
    for c in sorted(info["Containers"].values(), key=lambda c: c["Name"]):
        print(f"    {c['Name']:<45} {c['IPv4Address']}")
    print()

public_names = " ".join(c["Name"] for c in net_info["public_subnet"]["Containers"].values())
print("Backends on the public network:", "NONE (correct)" if "backend" not in public_names else "YES (wrong)")

public_subnet   172.28.1.0/24    internal=False
    multi_subnet_load_balancer-nginx-1            172.28.1.2/24

private_subnet  172.28.2.0/24    internal=True
    multi_subnet_load_balancer-backend1-1         172.28.2.2/24
    multi_subnet_load_balancer-backend2-1         172.28.2.3/24
    multi_subnet_load_balancer-nginx-1            172.28.2.4/24

Backends on the public network: NONE (correct)


**Expected:**
- `public_subnet` (`172.28.1.0/24`, `internal=False`): **nginx only**.
- `private_subnet` (`172.28.2.0/24`, `internal=True`): nginx, backend1 and backend2.
- "Backends on the public network: NONE": nginx is the only dual-homed container. ✅

## 4. QA 1: Load distribution (round-robin)

**Requirement:** send at least 10 consecutive requests and confirm the responses alternate evenly between backend 1 and backend 2.

**What this cell does:** sends 10 requests through the load balancer, prints who answered each one, then checks the split and the alternation.

In [5]:
results = send_requests(10)

counts = Counter(body for _, body, _ in results)
bodies = [body for _, body, _ in results]
alternates = all(bodies[i] != bodies[i + 1] for i in range(len(bodies) - 1))
print("\nDistribution:", dict(counts))
print("Strict alternation:", alternates)
print("QA 1:", "PASS" if len(counts) == 2 and set(counts.values()) == {5} and alternates else "CHECK")

  #  status     time  response
  1  200      0.060s  Request handled by backend1
  2  200      0.008s  Request handled by backend2
  3  200      0.004s  Request handled by backend1
  4  200      0.004s  Request handled by backend2
  5  200      0.005s  Request handled by backend1
  6  200      0.006s  Request handled by backend2
  7  200      0.005s  Request handled by backend1
  8  200      0.006s  Request handled by backend2
  9  200      0.005s  Request handled by backend1
 10  200      0.015s  Request handled by backend2

Distribution: {'Request handled by backend1': 5, 'Request handled by backend2': 5}
Strict alternation: True
QA 1: PASS


**Expected:** `backend1`, `backend2`, `backend1`… a **5 / 5** split with strict alternation → **PASS**.

This proves round-robin across stateless backends: no IP-hash, no sticky sessions, every request is free to go to either backend.

## 5. QA 2: High availability and failover

**Requirement:** stop one backend, confirm **100%** of requests reach the remaining backend with **zero dropped requests** (no 502), then restart it and confirm it rejoins the pool **automatically**.

### 5.1 Stop backend1 and keep sending traffic

**What this cell does:** stops the `backend1` container, then **immediately** sends 10 requests and counts the failures.

In [6]:
sh("docker", "compose", "stop", "backend1")

results = send_requests(10)

ok = sum(1 for status, body, _ in results if status == 200 and "backend2" in body)
dropped = len(results) - ok
print(f"\nServed by backend2: {ok}/10   Dropped or errors: {dropped}   Slowest: {max(s for *_, s in results):.2f}s")
print("QA 2 (failover):", "PASS" if dropped == 0 else "FAIL")

$ docker compose stop backend1
Container multi_subnet_load_balancer-backend1-1 Stopping 
 Container multi_subnet_load_balancer-backend1-1 Stopped

  #  status     time  response
  1  200      1.024s  Request handled by backend2
  2  200      0.014s  Request handled by backend2
  3  200      0.004s  Request handled by backend2
  4  200      0.006s  Request handled by backend2
  5  200      0.022s  Request handled by backend2
  6  200      0.014s  Request handled by backend2
  7  200      0.003s  Request handled by backend2
  8  200      0.003s  Request handled by backend2
  9  200      0.003s  Request handled by backend2
 10  200      0.003s  Request handled by backend2

Served by backend2: 10/10   Dropped or errors: 0   Slowest: 1.02s
QA 2 (failover): PASS


**Expected:** **10/10** answered by `backend2`, **0 dropped** → **PASS**.

- One request may take about **1 second**. nginx tried `backend1`'s old address, gave up after the 1-second `proxy_connect_timeout`, and **retried on `backend2`** (`proxy_next_upstream`). The client still got a normal 200.
- Compare version 1: the same test with the AI's config **dropped 2 of 5 requests** after 5-second hangs.

### 5.2 Restart backend1 and watch it rejoin

**What this cell does:** starts `backend1` again, waits until Docker reports it healthy, then sends requests until `backend1` answers. That shows it rejoined the round-robin pool **without restarting or reloading nginx**.

In [7]:
sh("docker", "compose", "start", "backend1")
t0 = time.time()

cid = container_id("backend1")
while sh("docker", "inspect", "-f", "{{.State.Health.Status}}", cid, show=False)[1] != "healthy" and time.time() - t0 < 60:
    time.sleep(1)
print(f"backend1 healthy after {time.time() - t0:.1f}s")

while "backend1" not in get()[1] and time.time() - t0 < 60:
    time.sleep(0.5)
print(f"backend1 back in rotation after {time.time() - t0:.1f}s (no nginx restart)\n")

results = send_requests(10)
counts = Counter(body for _, body, _ in results)
print("\nDistribution:", dict(counts))
print("QA 2 (rejoin):", "PASS" if len(counts) == 2 and all(s == 200 for s, _, _ in results) else "FAIL")

$ docker compose start backend1
Container multi_subnet_load_balancer-backend1-1 Starting 
 Container multi_subnet_load_balancer-backend1-1 Started

backend1 healthy after 6.6s
backend1 back in rotation after 7.7s (no nginx restart)

  #  status     time  response
  1  200      0.022s  Request handled by backend2
  2  200      0.016s  Request handled by backend1
  3  200      0.015s  Request handled by backend2
  4  200      0.003s  Request handled by backend1
  5  200      0.002s  Request handled by backend2
  6  200      0.002s  Request handled by backend1
  7  200      0.003s  Request handled by backend2
  8  200      0.003s  Request handled by backend1
  9  200      0.003s  Request handled by backend2
 10  200      0.003s  Request handled by backend1

Distribution: {'Request handled by backend2': 5, 'Request handled by backend1': 5}
QA 2 (rejoin): PASS


**Expected:** `backend1` is healthy after a few seconds, back in rotation after about **5–10 seconds**, and the next 10 requests split **5 / 5** again → **PASS**.

How it rejoins automatically: nginx re-checks Docker's DNS every 5 seconds (`resolver ... valid=5s` with `resolve`), so it finds `backend1` again as soon as the container is back. A failed backend is only paused for 5 seconds (`fail_timeout=5s`) before nginx tries it again.

## 6. QA 3: Network boundary and isolation (negative tests)

**Requirement:** prove that direct connections from the host to the backend ports are rejected, and that the backends can **only** be reached through the load balancer.

**What this cell does:** tries four things and reports each as expected or not:

| Test | From | To | Should |
|---|---|---|---|
| A | Host | `localhost:8000`, the backend's port | **Fail** |
| B | Host | backend1's private IP, port 8000 | **Fail** |
| C | nginx container | `backend1:8000`, over the private network | **Succeed** |
| D | backend1 container | the internet (`example.com`) | **Fail** (`internal: true`) |

In [8]:
b1_ip = next(c["IPv4Address"].split("/")[0] for c in
             json.loads(sh("docker", "network", "inspect", net_info["private_subnet"]["Name"], show=False)[1])[0]["Containers"].values()
             if "backend1" in c["Name"])
checks = []

for label, url, counts in [("A  host -> localhost:8000", "http://localhost:8000/", True),
                           (f"B  host -> {b1_ip}:8000", f"http://{b1_ip}:8000/", False)]:
    status, body, secs, _ = get(url=url, timeout=3)
    blocked = status == "FAILED"
    if counts:
        checks.append(blocked)
    note = "" if counts else "   (informational, depends on platform)"
    print(f"{label:<34} {'BLOCKED (' + body + ')' if blocked else 'REACHABLE: ' + body}{note}")

rc, out = sh("docker", "compose", "exec", "-T", "nginx", "wget", "-qO-", "-T", "3", "http://backend1:8000/", show=False)
checks.append(rc == 0 and "backend1" in out)
print(f"{'C  nginx -> backend1:8000':<34} {'REACHABLE: ' + out if rc == 0 else 'FAILED'}")

rc, out = sh("docker", "compose", "exec", "-T", "backend1", "python", "-c",
             "import urllib.request; urllib.request.urlopen('http://example.com', timeout=3)", show=False)
checks.append(rc != 0)
print(f"{'D  backend1 -> internet':<34} {'BLOCKED (' + out.splitlines()[-1][:60] + ')' if rc != 0 else 'REACHABLE'}")

print("\nQA 3:", "PASS" if all(checks) else "FAIL")

A  host -> localhost:8000          BLOCKED (URLError)
B  host -> 172.28.2.2:8000         BLOCKED (URLError)   (informational, depends on platform)
C  nginx -> backend1:8000          REACHABLE: Request handled by backend1
D  backend1 -> internet            BLOCKED (urllib.error.URLError: <urlopen error [Errno -3] Try again>)

QA 3: PASS


**Expected:**
- **A: BLOCKED.** Nothing listens on the host's port 8000, because the backends have no published ports. ✅
- **B: BLOCKED.** On Docker Desktop (Windows/macOS), containers run inside a virtual machine, so the host has no route to `172.28.2.x`. ✅
- **C: REACHABLE**, `Request handled by backend1`. Inside the private network nginx *can* reach the backend, so the load balancer is the only path in. ✅
- **D: BLOCKED.** `internal: true` gives the private network no route to the internet, so even a compromised backend can't call out. ✅

*Note:* test B is informational and doesn't count toward PASS. On a Linux machine running Docker natively it can show **REACHABLE**, because the Linux host is itself attached to Docker's bridge networks. Test A, the published-ports check the assignment asks about, is blocked on every platform.

## 7. QA 4: Health checks, headers and browser console

**What this cell does:**
1. Calls the **load balancer's** health check, `/healthz`, and prints the full response headers.
2. Calls **`/healthz/backend`**, which checks a backend's health *through* the load balancer. `X-Upstream-Server` shows which backend answered.
3. Shows Docker's own health status for each container.

In [9]:
status, body, secs, headers = get("/healthz")
print(f"GET /healthz -> {status} {body!r}")
for k, v in headers.items():
    print(f"    {k}: {v}")
ct_count = len(headers.get_all("Content-Type") or [])
print(f"    Content-Type headers: {ct_count}  {'(correct)' if ct_count == 1 else '(duplicate!)'}\n")

for _ in range(2):
    s, b, t, h = get("/healthz/backend")
    print(f"GET /healthz/backend -> {s} {b!r}   X-Upstream-Server: {h.get('X-Upstream-Server')}")

print()
sh("docker", "compose", "ps", "--format", "table {{.Service}}\t{{.Status}}")
print("QA 4:", "PASS" if status == 200 and ct_count == 1 and s == 200 else "CHECK")

GET /healthz -> 200 'nginx healthy'
    Server: nginx/1.31.8
    Date: Wed, 28 Oct 2026 20:19:08 GMT
    Content-Type: text/plain
    Content-Length: 14
    Connection: close
    Content-Type headers: 1  (correct)

GET /healthz/backend -> 200 'healthy'   X-Upstream-Server: 172.28.2.3:8000
GET /healthz/backend -> 200 'healthy'   X-Upstream-Server: 172.28.2.2:8000

$ docker compose ps --format table {{.Service}}	{{.Status}}
SERVICE    STATUS
backend1   Up 21 seconds (healthy)
backend2   Up 32 seconds (healthy)
nginx      Up 25 seconds

QA 4: PASS


**Expected:**
- `/healthz` → **200 OK**, `nginx healthy`, with **one** `Content-Type` header. The AI's draft sent two.
- `/healthz/backend` → **200** `healthy`, answered by a different backend IP each time (round-robin).
- Both backends show **(healthy)** in Docker.

### 7.1 Browser DevTools check (do this live)

1. Open **http://localhost:8080** in Chrome or Edge.
2. Press **F12** and open the **Network** tab. Tick **Disable cache**.
3. Refresh the page about 10 times. The page text alternates between `backend1` and `backend2`.
4. Click one request and show:
   - **Status:** `200 OK`
   - **Response headers:** `X-Upstream-Server: 172.28.2.x:8000`, which changes between the two backends, and `Connection: keep-alive` (connection reuse)
5. Open the **Console** tab: it should be **empty, with no errors**. `/favicon.ico` is answered by nginx with `204 No Content`, so the browser logs no 404 and the favicon request doesn't take a turn in the round-robin rotation.

## 8. Proof in the logs

**Requirement:** inspect container logs to prove the load balancer handled the health checks and redirected traffic.

**What this cell does:** reads nginx's access log. The corrected config logs **which backend** handled each request (`upstream=`) and that backend's status. When a request was retried, both backends appear, for example `upstream=172.28.2.2:8000, 172.28.2.3:8000 upstream_status=504, 200`.

In [10]:
_, logs = sh("docker", "compose", "logs", "--no-log-prefix", "nginx", show=False)
access = [l for l in logs.splitlines() if "upstream=" in l]
retries = [l for l in access if "," in l.split("upstream=")[1].split(" upstream_status")[0]]
errors = [l for l in logs.splitlines() if "[error]" in l or "[warn]" in l]

print(f"=== Requests whose traffic was REDIRECTED to the other backend ({len(retries)}) ===")
print("\n".join(retries[-5:]) or "(none)")
print(f"\n=== nginx detecting the stopped backend ({len(errors)}) ===")
print("\n".join(e[:180] for e in errors[-4:]) or "(none)")
print(f"\n=== Last 8 requests ===")
print("\n".join(access[-8:]))

=== Requests whose traffic was REDIRECTED to the other backend (1) ===
172.28.1.1 "GET / HTTP/1.1" 200 upstream=172.28.2.2:8000, 172.28.2.3:8000 upstream_status=504, 200 time=1.001s

=== nginx detecting the stopped backend (2) ===
2026/10/28 20:18:46 [warn] 30#30: *22 upstream server temporarily disabled while connecting to upstream, client: 172.28.1.1, server: , request: "GET / HTTP/1.1", upstream: "http://
2026/10/28 20:18:46 [error] 30#30: *22 upstream timed out (110: Operation timed out) while connecting to upstream, client: 172.28.1.1, server: , request: "GET / HTTP/1.1", upstream

=== Last 8 requests ===
172.28.1.1 "GET / HTTP/1.1" 200 upstream=172.28.2.2:8000 upstream_status=200 time=0.000s
172.28.1.1 "GET / HTTP/1.1" 200 upstream=172.28.2.3:8000 upstream_status=200 time=0.001s
172.28.1.1 "GET / HTTP/1.1" 200 upstream=172.28.2.2:8000 upstream_status=200 time=0.001s
172.28.1.1 "GET / HTTP/1.1" 200 upstream=172.28.2.3:8000 upstream_status=200 time=0.000s
172.28.1.1 "GET / HTTP/1.1

**Expected:**
- **Redirected requests:** one or more lines with **two** upstream addresses and `upstream_status=504, 200`. That is the request nginx retried on the healthy backend during QA 2.
- **Detection:** `upstream timed out ... while connecting to upstream` and `upstream server temporarily disabled`: nginx noticed the stopped backend and took it out of rotation. `backend1 could not be resolved` means Docker's DNS confirmed the container was gone.
- **Last requests:** `upstream=` alternates between the two backend addresses, and `/healthz/backend` requests appear too.

## 9. QA test summary

| # | QA test | Result | Evidence |
|---|---|---|---|
| 1 | **Round-robin:** 10 requests alternate evenly | ☐ PASS | Section 4: 5 / 5 split, strict alternation |
| 2a | **Failover:** 100% served by the healthy backend, 0 dropped, no 502 | ☐ PASS | Section 5.1: 10/10 by backend2 |
| 2b | **Rejoin:** restarted backend re-enters the pool automatically | ☐ PASS | Section 5.2: back in rotation with no nginx restart |
| 3 | **Isolation:** host cannot reach backend ports; only nginx can | ☐ PASS | Section 6: A, B, D blocked; C reachable |
| 4 | **Health and console:** `/healthz` 200 OK, clean headers, clean console | ☐ PASS | Section 7 and the browser DevTools check |

*(Tick each box after your test run: change ☐ to ☑.)*

### Before vs after

| | AI's draft (version 1) | Corrected (this notebook) |
|---|---|---|
| Requests dropped with one backend stopped | **2 of 5** (5-second hangs) | **0 of 10** (worst case about 1 s) |
| Stopped backend rejoins | Not tested; addresses resolved only at startup | Automatically, in about 5–10 s |
| `/healthz` | nginx only, duplicate header | nginx, plus `/healthz/backend` and Docker health checks |
| Private network | Not internal | `internal: true`, no internet route |

## 10. Takeaways: the load balancer as accelerator *and* perimeter

**Performance accelerator**
- Spreads requests evenly across replicas (round-robin), so capacity grows by adding backends.
- Hides failures: fast timeouts plus `proxy_next_upstream` turn a dead backend into a 1-second retry instead of a dropped request.
- Health-aware: failed backends are taken out of rotation and added back automatically.

**Security perimeter**
- The only published port. The backends have no host ports and no public network interface.
- Dual-homed: it is the single bridge between the public and private subnets.
- The private subnet is `internal`, so backends can't reach the internet even if compromised.

**Lesson from the AI audit:** the AI produced the right shape but not the failure handling. Its draft passed a "does it work?" test and failed the "does it keep working?" test. Only the failover test exposed the missing timeouts and retries.

## 11. Clean up (after recording)

**What this cell does:** stops and removes the containers and both networks. The project files and the built image stay. Run it only when you're finished.

In [11]:
_ = sh("docker", "compose", "down")

$ docker compose down
Container multi_subnet_load_balancer-nginx-1 Stopping 
 Container multi_subnet_load_balancer-nginx-1 Stopped 
 Container multi_subnet_load_balancer-nginx-1 Removing 
 Container multi_subnet_load_balancer-nginx-1 Removed 
 Container multi_subnet_load_balancer-backend2-1 Stopping 
 Container multi_subnet_load_balancer-backend1-1 Stopping 
 Container multi_subnet_load_balancer-backend2-1 Stopped 
 Container multi_subnet_load_balancer-backend2-1 Removing 
 Container multi_subnet_load_balancer-backend2-1 Removed 
 Container multi_subnet_load_balancer-backend1-1 Stopped 
 Container multi_subnet_load_balancer-backend1-1 Removing 
 Container multi_subnet_load_balancer-backend1-1 Removed 
 Network multi_subnet_load_balancer_private_subnet Removing 
 Network multi_subnet_load_balancer_public_subnet Removing 
 Network multi_subnet_load_balancer_public_subnet Removed 
 Network multi_subnet_load_balancer_private_subnet Removed



(0,
 'Container multi_subnet_load_balancer-nginx-1 Stopping \n Container multi_subnet_load_balancer-nginx-1 Stopped \n Container multi_subnet_load_balancer-nginx-1 Removing \n Container multi_subnet_load_balancer-nginx-1 Removed \n Container multi_subnet_load_balancer-backend2-1 Stopping \n Container multi_subnet_load_balancer-backend1-1 Stopping \n Container multi_subnet_load_balancer-backend2-1 Stopped \n Container multi_subnet_load_balancer-backend2-1 Removing \n Container multi_subnet_load_balancer-backend2-1 Removed \n Container multi_subnet_load_balancer-backend1-1 Stopped \n Container multi_subnet_load_balancer-backend1-1 Removing \n Container multi_subnet_load_balancer-backend1-1 Removed \n Network multi_subnet_load_balancer_private_subnet Removing \n Network multi_subnet_load_balancer_public_subnet Removing \n Network multi_subnet_load_balancer_public_subnet Removed \n Network multi_subnet_load_balancer_private_subnet Removed')